In [ ]:
# ============= 1. إنشاء env و تثبيت المكتبات =============

import os
import subprocess
import sys



env_name = "env-flan-t5"
print(f"🔧 جاري إنشاء البيئة الافتراضية '{env_name}'...")


subprocess.check_call([sys.executable, "-m", "venv", env_name])



if os.name == 'nt':  # Windows
    pip_path = os.path.join(env_name, "Scripts", "pip.exe")
else:  # Unix/Linux
    pip_path = os.path.join(env_name, "bin", "pip")



requirements_file = "requirements-flan-t5.txt"

subprocess.check_call([pip_path, "install", "-r", requirements_file])





In [6]:

import pandas as pd
import torch
import re
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    pipeline
)
# import evaluate
from tqdm import tqdm
import random

In [2]:
# ============= 2. تحميل وتنظيف البيانات =============
import pandas as pd
import ast

print("📥 جاري تحميل البيانات...")
df = pd.read_csv("recipes.csv")


# الاحتفاظ بالأعمدة المهمة فقط
essential_cols = ["Name", "Calories", "ProteinContent", "RecipeIngredientParts",
                  "RecipeInstructions", "FatContent", "CarbohydrateContent"]
df = df[essential_cols].dropna()

# تحليل القوائم النصية
def parse_list(x):
    try:
        if isinstance(x, str) and x.startswith('c'):
            x = x[1:]
        # إزالة الأقواس والاقتباسات للحصول على نص نظيف
        x = x.strip('()').replace('"', '').replace("'", '')
        return x
    except:
        return str(x)

df["RecipeIngredientParts"] = df["RecipeIngredientParts"].apply(parse_list)
df["RecipeInstructions"] = df["RecipeInstructions"].apply(parse_list)

# تحويل الأنواع
df["Calories"] = pd.to_numeric(df["Calories"], errors="coerce")
df["ProteinContent"] = pd.to_numeric(df["ProteinContent"], errors="coerce")
df["FatContent"] = pd.to_numeric(df["FatContent"], errors="coerce")
df["CarbohydrateContent"] = pd.to_numeric(df["CarbohydrateContent"], errors="coerce")

# إزالة القيم غير الصالحة
df = df.dropna()


📥 جاري تحميل البيانات...


C:\Users\IFIX\AppData\Local\Temp\ipykernel_14408\3084722079.py:6: DtypeWarning: Columns (0,2,14,15,16,17,18,19,20,21,22,23,24,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("recipes.csv")


In [17]:
# df.head()

In [4]:
# ============= 3. تنظيف النصوص =============
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # إزالة المسافات الزائدة
    text = re.sub(r'[^\w\s.,!?-]', '', text)  # إزالة الرموز غير المرغوبة
    return text.strip()

for col in ["Name", "RecipeIngredientParts", "RecipeInstructions"]:
    df[col] = df[col].apply(clean_text)


In [19]:
# df.head()

In [3]:

# ============= 4. تحديد هدف التغذية =============
def infer_nutrition_goal(row):
    calories = row["Calories"]
    protein = row["ProteinContent"]
    carbs = row["CarbohydrateContent"]
    fat = row["FatContent"]

    if calories < 400:
        goal = "Weight Loss"
        reason = f"Low in calories ({calories:.0f} kcal)"
    elif protein >= 25:
        goal = "Muscle Building"
        reason = f"High in protein ({protein:.1f} grams)"
    elif carbs < 30 and fat < 15:
        goal = "Keto Diet"
        reason = f"Low in carbohydrates ({carbs:.1f} grams)"
    elif fat < 10:
        goal = "Low-Fat Diet"
        reason = f"Low in fat ({fat:.1f} grams)"
    else:
        goal = "Balanced Healthy Diet"
        reason = "Good balance of nutrients"

    return goal, reason


df["Goal"], df["Goal_Reason"] = zip(*df.apply(infer_nutrition_goal, axis=1))



In [5]:
len(df)

522516

In [18]:
# ============= 5. بناء مجموعة بيانات المحادثة =============

def build_conversation_dataset(df, max_samples=20000):
    conversations = []

    system_prompt = """You are a professional nutritionist specialized in providing dietary advice.
Your tasks:
1. Provide personalized nutrition recommendations
2. Explain nutritional benefits clearly
3. Design daily and weekly meal plans
4. Suggest healthy alternatives
Be supportive, motivating, and accurate."""

    openers = [
        "Sure, let's take a look.",
        "That's a great goal to focus on.",
        "You're on the right track.",
        "Absolutely, here's a solid option for you.",
        "Good question — nutrition plays a big role here.",
    ]

    closings = [
        "Let me know if you'd like a lighter or higher-protein option.",
        "Portion size can be adjusted based on your calorie needs.",
        "I can also create a full weekly plan if you want.",
        "Consistency matters more than perfection — keep going!",
    ]

    for idx, row in tqdm(
        df.iterrows(), total=min(len(df), max_samples), desc="Building Conversations"
    ):
        if idx >= max_samples:
            break

        opener = random.choice(openers)
        closing = random.choice(closings)

        # ===============================
        # 1️⃣ Single meal recommendation
        # ===============================
        scenarios = [
            {
                "user": f"I need a meal for {row['Goal']}. Can you recommend something?",
                "assistant": f"""{opener}

**{row['Name']}** is a great option for your {row['Goal'].lower()} goal.

Nutrition per serving:
- Calories: {row['Calories']:.0f}
- Protein: {row['ProteinContent']:.1f}g
- Carbs: {row['CarbohydrateContent']:.1f}g
- Fat: {row['FatContent']:.1f}g

Why it works:
{row['Goal_Reason']}.

{closing}
"""
            },

        # ===============================
        # 2️⃣ Nutritional analysis
        # ===============================
            {
                "user": f"Tell me about the nutritional value of {row['Name']}",
                "assistant": f"""Here’s a nutritional breakdown of **{row['Name']}**:

- Energy: {row['Calories']:.0f} kcal
- Protein: {row['ProteinContent']:.1f}g (supports muscle and recovery)
- Carbohydrates: {row['CarbohydrateContent']:.1f}g (main energy source)
- Fat: {row['FatContent']:.1f}g

This meal fits well for **{row['Goal']}** because {row['Goal_Reason'].lower()}.
"""
            },

        # ===============================
        # 3️⃣ Preparation method
        # ===============================
            {
                "user": f"How can I prepare {row['Name']} for my {row['Goal']} diet?",
                "assistant": f"""To prepare **{row['Name']}** for your {row['Goal'].lower()} diet:

Ingredients:
{row['RecipeIngredientParts']}

Instructions:
{row['RecipeInstructions'][:500]}...

This preparation keeps calories controlled while preserving nutritional value.
"""
            },

        # ===============================
        # 4️⃣ Healthy meal list
        # ===============================
            {
                "user": f"Give me a list of healthy meals suitable for {row['Goal']}",
                "assistant": f"""Here are some healthy meal options suitable for {row['Goal'].lower()}:

• {row['Name']}
• High-protein lean meals
• Vegetable-rich balanced dishes
• Whole-grain based meals
• Light dinners with healthy fats

These options focus on nutrient density and long-term sustainability.
"""
            },

        # ===============================
        # 5️⃣ Daily meal plan
        # ===============================
            {
                "user": f"Create a daily meal plan for {row['Goal']}",
                "assistant": f"""Here’s a simple daily meal plan for {row['Goal'].lower()}:

Breakfast:
Light protein-rich meal (e.g. eggs, yogurt, oats)

Lunch:
**{row['Name']}** with balanced portions

Dinner:
Vegetables + lean protein (low-calorie, easy to digest)

This structure helps maintain energy while supporting your goal.
"""
            },

        # ===============================
        # 6️⃣ Weekly meal plan
        # ===============================
            {
                "user": f"Create a 7-day meal plan for {row['Goal']} including breakfast, lunch, and dinner",
                "assistant": f"""Here is a sample 7-day meal plan designed for {row['Goal'].lower()}.

Day 1:
Breakfast: Light protein meal
Lunch: {row['Name']}
Dinner: Vegetables with lean protein

Day 2:
Breakfast: Oats or yogurt
Lunch: Balanced whole-food meal
Dinner: Light dinner option

Day 3–7:
Follow the same structure while rotating protein sources and vegetables.

This approach ensures balance, variety, and consistency without complexity.
"""
            },
        ]

        for scenario in scenarios:
            conversations.append({
                "input_text": f"<system>{system_prompt}</system>\n<user>{scenario['user']}</user>",
                "target_text": scenario["assistant"]
            })

    return Dataset.from_list(conversations)


print("📊 جاري بناء مجموعة البيانات...")
dataset = build_conversation_dataset(
    df, max_samples=1000000
)  # تحديد الحد الأقصى لعدد العينات 2000 بدلاً من 10000 للتجريب


print(f"dataset: {len(dataset)}.  ")

📊 جاري بناء مجموعة البيانات...


Building Conversations: 100%|██████████| 522516/522516 [01:07<00:00, 7713.73it/s]


dataset: 3135096.  


In [20]:
# print(len(dataset))
# dataset[:10]


3135096


{'input_text': ['<system>You are a professional nutritionist specialized in providing dietary advice.\nYour tasks:\n1. Provide personalized nutrition recommendations\n2. Explain nutritional benefits clearly\n3. Design daily and weekly meal plans\n4. Suggest healthy alternatives\nBe supportive, motivating, and accurate.</system>\n<user>I need a meal for Weight Loss. Can you recommend something?</user>',
  '<system>You are a professional nutritionist specialized in providing dietary advice.\nYour tasks:\n1. Provide personalized nutrition recommendations\n2. Explain nutritional benefits clearly\n3. Design daily and weekly meal plans\n4. Suggest healthy alternatives\nBe supportive, motivating, and accurate.</system>\n<user>Tell me about the nutritional value of Low-Fat Berry Blue Frozen Dessert</user>',
  '<system>You are a professional nutritionist specialized in providing dietary advice.\nYour tasks:\n1. Provide personalized nutrition recommendations\n2. Explain nutritional benefits clea

In [21]:

# ============= 6. تقسيم البيانات =============
print("✂️ جاري تقسيم البيانات...")
split_dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"📈 حجم البيانات: التدريب = {len(train_dataset)}، التقييم = {len(eval_dataset)}")



✂️ جاري تقسيم البيانات...
📈 حجم البيانات: التدريب = 2821586، التقييم = 313510


In [24]:
# # ============= 6.5. تقليل حجم البيانات قبل Tokenization =============
# print("📉 جاري تقليل حجم البيانات للذاكرة...")

# # تقليل حجم بيانات التدريب إلى 4000 فقط
# train_dataset = train_dataset.select(range(min(4000, len(train_dataset))))

# # تقليل حجم بيانات التقييم إلى 200 فقط
# eval_dataset = eval_dataset.select(range(min(200, len(eval_dataset))))

# print(f"📊 الحجم بعد التقليل: التدريب = {len(train_dataset)}، التقييم = {len(eval_dataset)}")

In [26]:

# ============= 7. تحميل النموذج والTokenizer =============
print("🤖 جاري تحميل النموذج...")
MODEL_NAME = "google/flan-t5-small"  # حجم أكبر من small لأداء أفضل

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    force_download=True,  # إجبار التحميل من جديد
    resume_download=True,  # استئناف التحميل إذا كان هناك انقطاع
    local_files_only=False  # لا تعتمد على الملفات المحلية فقط
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    force_download=True,
    resume_download=True,
    local_files_only=False
)
    


# إضافة رموز خاصة للمحادثة
special_tokens = ["<system>", "<user>", "<assistant>"]
tokenizer.add_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))


🤖 جاري تحميل النموذج...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Embedding(32103, 512)

In [24]:
# ============= 8. Tokenization مع طول أقل =============
def preprocess_function(examples, max_length=256):
    inputs = examples["input_text"]
    targets = examples["target_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        truncation=True,
        padding="max_length",
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=128,
            truncation=True,
            padding="max_length",
        )

    model_inputs["labels"] = labels["input_ids"]
    
    # تحويل -100 لـ pad_token_id في labels
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in model_inputs["labels"]
    ]
    
    return model_inputs




print("🔤 جاري معالجة النصوص...")
train_tokenized = train_dataset.map(
    preprocess_function,
    batched=True,
     batch_size=500,  # ⭐ تقليل حجم الدفعة
    remove_columns=train_dataset.column_names
)

eval_tokenized = eval_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=200,  # ⭐ تقليل حجم الدفعة
    remove_columns=eval_dataset.column_names
)

🔤 جاري معالجة النصوص...


Map:   0%|          | 0/2821586 [00:00<?, ? examples/s]


NameError: name 'tokenizer' is not defined

In [ ]:
# ============= 9. إعداد معايير التدريب =============
# ============= باختصار =============
print("⚙️  إعداد التدريب...")

training_args = TrainingArguments(
    output_dir="./nutrition_chatbot_v2",
    evaluation_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="steps",  # حفظ checkpoints كل عدد من الخطوات
    save_steps=500,  # حفظ كل 500 خطوة
    save_total_limit=3,  # الاحتفاظ بآخر 3 checkpoints فقط
    fp16=torch.cuda.is_available(),  # هنا نختبر GPU مباشرة
    dataloader_num_workers=0,
)

⚙️  إعداد التدريب...
✅ جاهز: دفعة 2, 3 عهود


In [ ]:
# print(f"before: train_tokenized: {train_tokenized}, eval_tokenized: {eval_tokenized}")


# # تقليل حجم البيانات بشكل عشوائي
# if len(train_tokenized) > 5000:
#     train_tokenized = train_tokenized.train_test_split(train_size=5000, seed=42)['train']
#     print(f"📉 تم تقليل بيانات التدريب إلى {len(train_tokenized)} عينة عشوائية.")
# if len(eval_tokenized) > 1000:
#     eval_tokenized = eval_tokenized.train_test_split(train_size=1000, seed=42)['train']
#     print(f"📉 تم تقليل بيانات التقييم إلى {len(eval_tokenized)} عينة عشوائية.")



In [23]:
# ============= 10. إعداد Data Collator =============
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)



NameError: name 'tokenizer' is not defined

In [30]:
import nltk
nltk.download("punkt")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\IFIX\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [31]:
# ============= 11. دقة التقييم - ROUGE فقط =============
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics_safe(eval_pred):
    """
    دالة آمنة لحساب المقاييس
    """
    try:
        predictions, labels = eval_pred
        
        # معالجة predictions
        if isinstance(predictions, tuple):
            predictions = predictions[0]
        
        # تحويل logits إلى token IDs
        if predictions.ndim == 3:
            predictions = np.argmax(predictions, axis=-1)
        
        # فك ترميز
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        
        # استبدال -100 في labels
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        # تصفية النصوص الفارغة
        filtered_preds = []
        filtered_labels = []
        
        for pred, label in zip(decoded_preds, decoded_labels):
            if pred.strip() and label.strip():  # فقط النصوص غير الفارغة
                filtered_preds.append(pred)
                filtered_labels.append(label)
        
        if not filtered_preds or not filtered_labels:
            print("⚠️ تحذير: لا توجد نصوص صالحة للحساب")
            return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
        
        # حساب ROUGE
        result = rouge.compute(
            predictions=filtered_preds,
            references=filtered_labels,
            use_stemmer=True
        )
        
        return {
            "rouge1": round(result["rouge1"], 4),
            "rouge2": round(result["rouge2"], 4),
            "rougeL": round(result["rougeL"], 4)
        }
        
    except Exception as e:
        print(f"⚠️ خطأ في compute_metrics: {e}")
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

In [22]:
# ============= 11.5 التحقق من البيانات =============
# فحص عينة من البيانات
sample = train_tokenized[0]
print("🔍 فحص عينة من البيانات:")
print(f"   keys: {sample.keys()}")
print(f"   input_ids length: {len(sample['input_ids'])}")
print(f"   attention_mask length: {len(sample['attention_mask'])}")
print(f"   labels length: {len(sample['labels'])}")

# فك ترميز الـ input_ids
input_text = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f"\n📝 النص المدخل (input_text):")
print(input_text[:200])

# فك ترميز الـ labels (مع استبدال -100 بـ tokenizer.pad_token_id)
labels = sample['labels']
labels = [label if label != -100 else tokenizer.pad_token_id for label in labels]
label_text = tokenizer.decode(labels, skip_special_tokens=False)
print(f"\n🎯 النص الهدف (label_text):")
print(label_text[:200])

# حساب عدد الـ labels الصالحة (غير -100)
num_valid_labels = sum(1 for label in sample['labels'] if label != -100)
print(f"\n📊 عدد الـ labels الصالحة (غير -100): {num_valid_labels} من أصل {len(sample['labels'])}")

NameError: name 'train_tokenized' is not defined

In [33]:
import numpy as np
print(np.__version__)


1.26.4


In [ ]:
# ============= 12. التدريب - معدل =============
import os
import json
import torch
import gc


# تعطيل accelerate DataLoader
import os
os.environ["ACCELERATE_DISABLE_RICH"] = "1"




# تنظيف الذاكرة
def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

clean_memory()

# إعداد Trainer المصحح
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,  
    eval_dataset=eval_tokenized,  
    data_collator=data_collator,
    compute_metrics=compute_metrics_safe,  # ✅ استخدام الدالة الصحيحة
)

print("🚀 بدء التدريب...")

try:
    # التدريب
    train_result = trainer.train()

    print("✅ انتهى التدريب بنجاح!")
    print(f"📊 الخسارة النهائية: {train_result.training_loss:.4f}")

    # حفظ النموذج النهائي
    output_dir = "./nutrition_chatbot_final"
    print(f"💾 جاري حفظ النموذج في {output_dir}...")
    
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    # حفظ معلمات التدريب
    training_info = {
        "training_loss": train_result.training_loss,
        "global_step": trainer.state.global_step,
        "epoch": trainer.state.epoch,
        "metrics": trainer.state.log_history[-1] if trainer.state.log_history else {}
    }
    
    with open(os.path.join(output_dir, "training_info.json"), "w") as f:
        json.dump(training_info, f, indent=2)
    
    print(f"✅ تم الحفظ بنجاح في: {output_dir}")
    
    # اختبار النموذج على عينة
    print("\n🧪 اختبار النموذج على عينة...")
    test_input = "<system>أنت خبير تغذية محترف.</system>\n<user>أريد وجبة صحية منخفضة السعرات</user>"
    
    inputs = tokenizer(test_input, return_tensors="pt", truncation=True, max_length=1024)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.7,
            do_sample=True
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"🤖 استجابة النموذج: {response[:200]}...")

except Exception as e:
    print(f"❌ حدث خطأ: {e}")
    import traceback
    traceback.print_exc()

🚀 بدء التدريب...



























                                               

                                         
  1%|          | 1/150 [01:10<05:53,  2.37s/it]



{'eval_loss': 2.046506881713867, 'eval_rouge1': 0.5166, 'eval_rouge2': 0.2416, 'eval_rougeL': 0.4527, 'eval_runtime': 8.4047, 'eval_samples_per_second': 2.975, 'eval_steps_per_second': 2.975, 'epoch': 1.0}


                                               
  1%|          | 1/150 [01:54<05:53,  2.37s/it]

{'loss': 2.5622, 'learning_rate': 6.666666666666667e-05, 'epoch': 2.0}



























                                               
                                            

  1%|          | 1/150 [02:02<05:53,  2.37s/it]



{'eval_loss': 1.748032569885254, 'eval_rouge1': 0.5515, 'eval_rouge2': 0.2922, 'eval_rougeL': 0.5012, 'eval_runtime': 7.7902, 'eval_samples_per_second': 3.209, 'eval_steps_per_second': 3.209, 'epoch': 2.0}



























                                               

                                         
  1%|          | 1/150 [02:52<05:53,  2.37s/it]

                                               
100%|██████████| 75/75 [02:38<00:00,  2.12s/it]


{'eval_loss': 1.6522146463394165, 'eval_rouge1': 0.565, 'eval_rouge2': 0.3062, 'eval_rougeL': 0.5151, 'eval_runtime': 7.7499, 'eval_samples_per_second': 3.226, 'eval_steps_per_second': 3.226, 'epoch': 3.0}
{'train_runtime': 158.7636, 'train_samples_per_second': 0.945, 'train_steps_per_second': 0.472, 'train_loss': 2.3627167765299477, 'epoch': 3.0}
✅ انتهى التدريب بنجاح!
📊 الخسارة النهائية: 2.3627
💾 جاري حفظ النموذج في ./nutrition_chatbot_final...
✅ تم الحفظ بنجاح في: ./nutrition_chatbot_final

🧪 اختبار النموذج على عينة...
🤖 استجابة النموذج:          ...


In [ ]:
# ============= اختبار النموذج المدرب =============
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("🔧 جاري تحميل النموذج المدرب...")

# تحميل النموذج والـ tokenizer المحفوظين
model_path = "./nutrition_chatbot_final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# نقل النموذج إلى الجهاز المناسب
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ تم تحميل النموذج على {device}")

# أمثلة للاختبار
test_cases = [
    {
        "input": "<system>You are a professional nutritionist specialized in providing dietary advice.</system>\n<user>I need a meal for Weight Loss. Can you recommend something?</user>",
        "description": "توصية وجبة لفقدان الوزن"
    },
    {
        "input": "<system>You are a professional nutritionist specialized in providing dietary advice.</system>\n<user>Tell me about the nutritional value of a healthy meal</user>",
        "description": "سؤال عن القيم الغذائية"
    },
    {
        "input": "<system>You are a professional nutritionist specialized in providing dietary advice.</system>\n<user>How can I prepare a muscle building meal?</user>",
        "description": "طريقة تحضير وجبة لبناء العضلات"
    }
]

print("\n🧪 بدء الاختبارات...")

for i, test_case in enumerate(test_cases, 1):
    print(f"\n--- اختبار {i}: {test_case['description']} ---")
    print(f"📝 المدخل: {test_case['input']}")

    # Tokenization
    inputs = tokenizer(
        test_case['input'],
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    # توليد الاستجابة
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            temperature=0.6,
            do_sample=True,
            top_p=0.85,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            num_beams=4,
            early_stopping=True
        )

    # فك الترميز
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"🤖 الاستجابة:")
    print(response)
    print("-" * 50)

print("\n✅ انتهى الاختبار!")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


🔧 جاري تحميل النموذج المدرب...
✅ تم تحميل النموذج على cpu

🧪 بدء الاختبارات...

--- اختبار 1: توصية وجبة لفقدان الوزن ---
📝 المدخل: <system>You are a professional nutritionist specialized in providing dietary advice.</system>
<user>I need a meal for Weight Loss. Can you recommend something?</user>
🤖 الاستجابة:
Great! Here's how to make it! **It's a great idea! **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:**  **What you'll need:** 
--------------------------------------------------

--- اختبار 2: سؤال عن القيم الغذائية ---
📝 المدخل: <system>You are a professional nutritionist specialized in providing dietary advice.</system>
<user>Tell me about the nutritional value of a healthy meal</user>
🤖 الاستجابة:
Great question! Let me break down the nutritional profile of 